# DIFFUSION MODELS

In [1]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST, QMNIST, KMNIST
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time
import json
from tqdm import tqdm  # import the class directly

import torchaudio.transforms as T
from src.dataset import NSynth
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion"
paths = {
    'minst' : {
        'root': root + r"\minst",
        'model': root + r"\minst\model.pth",
        'object': root + r"\minst\object.pth",
        'scheduler': root + r"\minst\scheduler.pth",
        'checkpoint': root + r"\minst\checkpoint.pth",
        'dataset' : r"C:\Users\Articuno\Desktop\TFG-info\data\mnist"
    },
    'audio' : {
        'root': root + r"\audio",
        'model': root + r"\audio\model.pth",
        'object': root + r"\audio\object.pth",
        'scheduler': root + r"\audio\scheduler.pth",
        'checkpoint': root + r"\audio\checkpoint.pth"
    },
    'latent' : {
        'root': root + r"\latent",
        'model': root + r"\latent\model.pth",
        'object': root + r"\latent\object.pth",
        'scheduler': root + r"\latent\scheduler.pth",
        'checkpoint': root + r"\latent\checkpoint.pth",
        'dataset_training' : r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion\latent\dataset\training.pt",
        'training_norm_params' :  r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion\latent\dataset\training_params.pt",
    }
}

def show_loss_plot(losses):
    epochs_list = [d['epoch'] for d in losses]
    loss_list   = [d['loss']  for d in losses]
    lr_list     = [d['lr']    for d in losses]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    ax1.plot(epochs_list, loss_list)
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True)

    ax2.plot(epochs_list, lr_list, color='orange')
    ax2.set_ylabel('Learning Rate')
    ax2.set_xlabel('Epoch')
    ax2.set_title('Learning Rate Schedule')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
def save_checkpoint(model, scheduler, optimizer, lr_scheduler, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'lr_scheduler_state_dict': lr_scheduler.state_dict(),
        'loss': loss,
    }, path)

def load_checkpoint(model, scheduler, optimizer, lr_scheduler, path):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

def train(
    model, 
    scheduler,
    transform_data=None,
    epochs=100, 
    batch_size=16, 
    lr=1e-3, 
    modality='audio',
    dataset=NSynth('training'), 
    verbose=False,
    verbose_batchs=0,
    resume=False
):
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    diffuser = LatentDiffuser(model, scheduler).to(device) if modality == 'latent' else Diffuser(model, scheduler).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    
    scaler = torch.cuda.amp.GradScaler()
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


    mse_loss = nn.MSELoss()
    best_loss = 1
    losses = []
    
    num_timesteps = scheduler.alpha_bar.shape[0]


    # Retomar desde checkpoint si existe
    if resume and os.path.exists(paths[modality]['checkpoint']):
        model, scheduler, optimizer, lr_scheduler, start_epoch, best_loss = load_checkpoint(
            model, scheduler, optimizer, lr_scheduler, paths[modality]['checkpoint']
        )
        start_epoch += 1  # continúa desde el siguiente epoch
        print(f"Retomando desde epoch {start_epoch}, mejor loss: {best_loss:.4f}")
        
    for epoch in range(epochs):
        start_time = time.time() 
        _d_batch_index = 0
        epoch_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        
        for x in pbar:
            _d_batch_index += 1
            # wave = wave.to(device)
            # x = stft_transform(wave)
            if transform_data is not None:
                x = transform_data(x)
            x = x.to(device)                
            B = x.size(0)
            t = torch.randint(0, num_timesteps, (B,), device=device, dtype=torch.long)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(dtype=torch.bfloat16): # OPTIMIZACION FLOAT 32
                # print(f"mag: {mag.shape}, sin: {sin.shape}, cos: {cos.shape}, x: {x.shape}")
                z, e = diffuser(x, t)
                e_pred = model(z, t)
                if e_pred.shape != e.shape: # TODO no entiendo muy bien que ocurre
                    print('Error en las shapes')
                    e_pred = e_pred.squeeze(1)  # [B, 1, H, W] → [B, H, W]
                loss = mse_loss(e_pred, e)
                epoch_loss += loss.item()

            scaler.scale(loss).backward() # TODO no se si la funcion de loss es la mas optima para el caso
            scaler.step(optimizer)

            scaler.update()
            
            pbar.set_postfix({'batch loss': loss.item(), 'avg loss': epoch_loss / _d_batch_index})


        avg_loss = epoch_loss / _d_batch_index
        
        losses.append({
            'epoch': epoch,
            'loss': epoch_loss / _d_batch_index,
            'lr': optimizer.param_groups[0]['lr']
        })        
        lr_scheduler.step() # LEARNING SCHEDULER
            
        if verbose:
            _t = time.time() - start_time
            # print(f"Epoch {epoch}, Loss: {loss.item()}, time: {time.time() - start_time}, avg x time: {_t/(batch_size*_d_batch_index)}")
            
        if avg_loss < best_loss:
            best_loss = avg_loss
            
            if modality:
                save_checkpoint(model, scheduler, optimizer, epoch, avg_loss, paths[modality]['checkpoint'])
            
    
    # AL FINALIZAR EL ENTRENAMIENTO, GUARDAMOS EL MODELO Y EL SCHEDULER
    if modality:
        torch.save(model.state_dict(), paths[modality]['model'])
        torch.save(scheduler.state_dict(), paths[modality]['scheduler'])
        torch.save(model, paths[modality]['object'])
        
    print("Training completed., best loss:", best_loss)
    
    return model, scheduler, losses
    

## MINST

In [3]:
def MINST_transform(x):
    img, _ = x
    return img

In [ ]:
def setup_minst_model(timesteps=1000, channels=32, norm_groups=6, emb_dim=128):
    ## Image Size
    # input_height = 28
    # input_width = 28
    # input_size = (input_height, input_width)

    down_layers = [  
        DummyLayer(channels,    channels*2,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c canales
        DummyLayer(channels*2,  channels*4,  norm_groups, emb_dim, skip=False, stride=1).to(device),  # skip: c*2 canales
        DummyLayer(channels*4,  channels*8,  norm_groups, emb_dim, skip=False, stride=1).to(device),  # skip: c*4 canales
        # DummyLayer(c*8,  c*16, norm_groups, emb_dim, skip=True, stride=2).to(device),  # skip: c*8 canales
    ]

    bottleneck = DummyLayer(channels*8, channels*8, norm_groups, emb_dim).to(device)

    up_layers = [
        # DummyLayer(c*16 + c*8,  c*8, norm_groups, emb_dim, stride=-2).to(device),
        DummyLayer(channels*8,  channels*4, norm_groups, emb_dim, stride=-1).to(device),
        DummyLayer(channels*4,  channels*2, norm_groups, emb_dim, stride=-1).to(device),
        DummyLayer(channels*2 + channels,    channels,   norm_groups, emb_dim, stride=-1).to(device),
    ]

    # EMBEDDER
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # SCHEDULER
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    # MODEL
    model = DiffusionModel(
        layer_channels=(channels, channels),
        norm_groups=norm_groups,
        up_layers=up_layers,
        down_layers=down_layers,
        bottleneck=bottleneck,
        embedder=embedder, 
        input_channels=1,
        output_channels=1,
    ).to(device)
    
    return model, scheduler, MINST_transform

def train_MINST(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=2048, 
    learning_rate=1e-4, 
    resume=False
    ):

    # %%time
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality='minst',
        dataset= QMNIST(root=paths['minst']['dataset'], train=True,  download=True, transform=ToTensor()),
        verbose=True,
        verbose_batchs=0,
        resume=resume
    )
    show_loss_plot(losses)
    
    return model, scheduler, losses


## AUDIO

In [5]:
class AudioPipeline:
    def __init__(self, stft_transform):
        self.stft_transform = stft_transform

    def __call__(self, x):
        wave, _, _, _ = x
        wave = wave.to(device)  # <-- mover wave a GPU antes de la STFT
        stft_spec = self.stft_transform(wave)
        del wave
        log_mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec)
        return torch.cat([log_mag, sin, cos], dim=1)  # ya está en device

In [6]:
def setup_audio_model(timesteps=1000, channels=32, norm_groups=6, emb_dim=12, n_fft=1500, hop_length=250, win_length=1500):
    ''' Devuelve el modelo de audio listo para entrenar'''
    
    sample_rate = 16000
    # n_fft = 1500 # DISMINUIR TAMAÑO PARA OPTIMIZAR
    # hop_length = 250
    # win_length = n_fft
    
    sample_rate = 16000
    # n_fft = 1024//2
    # hop_length = 256
    # win_length = 1024//2
    
    # Data pipeline
    stft_transform = T.Spectrogram(
        n_fft=n_fft,
        win_length=win_length, 
        hop_length=hop_length,
        power=None, 
        onesided=True,
        center=False
    ).to(device)
    
    
    transform_audio = AudioPipeline(stft_transform)
    
    # Layers
    down_layers = [  
        DummyLayer(channels,    channels*2,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c canales
        DummyLayer(channels*2,  channels*4,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c*2 canales
    ]

    bottleneck = DummyLayer(channels*4, channels*4, norm_groups, emb_dim).to(device)

    up_layers = [ # TODO Podria hacer esto con stride = -2? o coger y hacer que si el stride es negativo, aumente manualmente 
        DummyLayer(channels*4 + channels*2, channels*2, norm_groups, emb_dim, stride=1).to(device),
        DummyLayer(channels*2 + channels, channels, norm_groups, emb_dim, stride=1).to(device),
    ]

    # Embedder
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # scheduler
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    # model
    model = DiffusionModel(
        layer_channels=(channels, channels), # Por ahora los canales de entrada y salida son los mismos
        norm_groups=norm_groups,
        up_layers=up_layers,
        down_layers=down_layers,
        bottleneck=bottleneck,
        embedder=embedder, 
        input_channels=3, ##SINCOS
        output_channels=3, ##SINCOS
    ).to(device)
    
    transform_audio = AudioPipeline(stft_transform)
    
    return model, scheduler, transform_audio

def train_audio(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=128, 
    learning_rate=1e-4, 
    ):
    
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality='audio',
        dataset=NSynth('training'),
        verbose=True,
        verbose_batchs=100,
    )
    show_loss_plot(losses)
    print(f'learning_rate: {learning_rate}')
    return model, scheduler, losses

# LATENT SPACE
Entrenamos un modelo de difusión en el espacio latente de un VAE

In [7]:
def latent_pipeline(x):
    if isinstance(x, (list, tuple)):
        x = x[0]        # TensorDataset wraps in tuple → [B, 200]
    return x             # [B, 200] ← ya es la forma correcta para el MLP

In [8]:
import torch.nn as nn


def setup_latent_model(timesteps=1000, emb_dim=128, hidden_dims=32, latent_dim=200, deep=4, increase_dims=True):
    ''' Devuelve el modelo de audio listo para entrenar'''
       
    if isinstance(hidden_dims, int):
        if increase_dims:
            hidden_dims = [hidden_dims * (2 ** i) for i in range(deep)]  # [32, 64, 128, 256] por ejemplo
        else:
            hidden_dims = [hidden_dims for _ in range(deep)]  # [32, 32, 32, 32] por ejemplo
    
    if len(hidden_dims) != deep:
        print('errror')
        
    hidden_dims.insert(0, latent_dim)  # [200, 256, 512, 1024]
    deep = len(hidden_dims) - 1        # 3 bloques

    # Down: 200→256, 256→512, 512→1024
    blocks_down = nn.ModuleList([
        nn.Sequential(
            nn.Linear(hidden_dims[i], hidden_dims[i+1]),
            nn.SiLU(),
            nn.Linear(hidden_dims[i+1], hidden_dims[i+1]),
        ) for i in range(deep)
    ])
    norms_down = nn.ModuleList([
        nn.LayerNorm(hidden_dims[i]) for i in range(deep)
    ])

    # Up: (1024+512)→512, (512+256)→256, (256+200)→200
    blocks_up = nn.ModuleList([
        nn.Sequential(
            nn.Linear(hidden_dims[i+1] + hidden_dims[i], hidden_dims[i]),
            nn.SiLU(),
            nn.Linear(hidden_dims[i], hidden_dims[i]),
        ) for i in reversed(range(deep))
    ])
    norms_up = nn.ModuleList([
        nn.LayerNorm(hidden_dims[i+1] + hidden_dims[i]) for i in reversed(range(deep))
    ])
    
    # Bottleneck
    bottleneck = BottleneckLatent(hidden_dims[-1], hidden_dims[-1]).to(device)

    # Embedder
    embedder = Embedder(num_timesteps=timesteps, embed_dim=emb_dim).to(device)

    # scheduler
    scheduler = Scheduler(num_timesteps=timesteps).to(device)

    # model
    model = LatentDiffusionMLP(
        latent_dim=latent_dim,
        hidden_dims=hidden_dims,
        blocks_up=blocks_up,
        blocks_down=blocks_down,
        norms_up=norms_up,
        norms_down=norms_down,
        embedder=embedder,
        bottleneck=bottleneck,
    ).to(device)
    
    return model, scheduler, latent_pipeline

def train_latent(
    model,
    scheduler,
    transform_data=None,
    epochs=10, 
    batch_size=128, 
    learning_rate=1e-4, 
    ):
    
    latents = torch.load(rf"{paths['latent']['dataset_training']}")
        
    # Calcula media y std del dataset de latentes
    mean = latents.mean()
    std = latents.std()
    # latents = (latents - mean) / std  # que tengan distribución ~N(0,1)
    
    model, scheduler, losses = train(
        model, 
        scheduler,
        transform_data=transform_data,
        epochs=epochs,
        batch_size=batch_size, 
        lr=learning_rate, 
        modality='latent',
        dataset=TensorDataset(latents),
        verbose=True,
        verbose_batchs=100,
    )
    show_loss_plot(losses)
    print(f'learning_rate: {learning_rate}')
    return model, scheduler, losses

# MODEL LOADING OR TRAINING

In [ ]:
''' 
ENTRENAMIENTO DEL MODELO MINST
    se pueden configurar más elementos directamente en setup_minst_model 

'''
from sklearn.utils import resample


TRAIN = False
CHECKPOINT = False

model, scheduler, trans = setup_minst_model(
    timesteps=300, # Tiene que ser el mismo que la que tuviera el modelo
    channels=64,
    norm_groups=8,
    emb_dim=128
)

if TRAIN:
    model, scheduler, losses = train_MINST(
        model=model,
        scheduler=scheduler,
        transform_data=trans,
        epochs=20,
        batch_size=2048//8,
        learning_rate=1e-4,
        resume=CHECKPOINT
    )
else:
    # model = torch.load(paths['minst']['object'])
    model.load_state_dict(torch.load(paths['minst']['model']))
    scheduler.load_state_dict(torch.load(paths['minst']['scheduler']))


''' 
ENTRENAMIENTO DEL MODELO AUDIO
    se pueden configurar más elementos directamente en setup_audio_model 

'''
TRAIN = False
CHECKPOINT = True

model, scheduler, trans = setup_audio_model(
    timesteps=750, # Tiene que ser el mismo que la que tuviera el modelo
    channels=32*4,
    norm_groups=8*4,
    emb_dim=128*4,
    n_fft = 1024//2,
    hop_length = 1024//4,
    win_length = 1024//2,
)

if CHECKPOINT:
    model, scheduler, _, _, _ = load_checkpoint(model, scheduler, None, paths['audio']['checkpoint'])

if TRAIN:
    model, scheduler, losses = train_audio(
        model=model,
        scheduler=scheduler,
        transform_data=trans,
        epochs=5,
        batch_size=8,
        learning_rate=1e-4
    )
else:
    # model = torch.load(paths['minst']['object'])
    model.state_dict(torch.load(paths['audio']['model']))
    scheduler.load_state_dict(torch.load(paths['audio']['scheduler']))


In [ ]:
''' 
ENTRENAMIENTO DEL MODELO LATENT DIFFUSION
    se pueden configurar más elementos directamente en setup_latent_model 

'''
TRAIN = True

model, scheduler, trans = setup_latent_model(
    timesteps=1000, 
    emb_dim=256, 
    hidden_dims=512, 
    latent_dim=200, # siempre es 200 porque es el tamaño del vector del VAE
    deep=3, 
    increase_dims=False
)

if TRAIN:
    model, scheduler, losses = train_latent(
        model=model,
        scheduler=scheduler,
        transform_data=trans,
        epochs=20,
        batch_size=2048//16,
        learning_rate=1e-4
    )
else:
    # model = torch.load(paths['minst']['object'])
    model.state_dict(torch.load(paths['latent']['model']))
    scheduler.load_state_dict(torch.load(paths['latent']['scheduler']))


Epoch 9/20:  10%|█         | 230/2260 [00:03<00:27, 72.86batch/s, batch loss=0.317, avg loss=0.298]

# SAMPLING

In [ ]:
def sample_raw_img(model, scheduler, image_size=(1, 28, 28), num_images=1):
    model.eval() # NS
    # with torch.no_grad():
    
    x = torch.randn(num_images, *image_size, device=device) # imagen de ruidio inicial
    T = scheduler.alpha_bar.shape[0]
    
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar
    with torch.no_grad():

        for t in reversed(range(T)):
            t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)

            # predicción de ruido
            e_pred = model(x, t_batch)
            
            beta_t = betas[t]
            alpha_t = alphas[t]
            alpha_bar_t = alpha_bars[t]
            
            # beta_t = beta_t.view(1,1,1,1)
            # alpha_t = alpha_t.view(1,1,1,1)
            # alpha_bar_t = alpha_bar_t.view(1,1,1,1)


            # Coeficientes DDPM
            coef1 = 1 / torch.sqrt(alpha_t)
            coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

            mu = coef1 * (x - coef2 * e_pred)

            if t > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mu + sigma_t * noise
            else:
                x = mu

        x = x.clamp(0, 1).cpu()
        
    return x

def denoise_img(model, scheduler, noisy_img, num_steps=300, step_imgs=10):
    model.eval()
    xs = []
    x = noisy_img.to(device)
    T = scheduler.alpha_bar.shape[0]
    
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar
    
    with torch.no_grad():
        for t in reversed(range(T)):
            t_batch = torch.full((x.size(0),), t, device=device, dtype=torch.long)

            e_pred = model(x, t_batch)
            
            beta_t = betas[t]
            alpha_t = alphas[t]
            alpha_bar_t = alpha_bars[t]

            coef1 = 1 / torch.sqrt(alpha_t)
            coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

            mu = coef1 * (x - coef2 * e_pred)

            if t > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mu + sigma_t * noise
            else:
                x = mu
                
            if t % (T // step_imgs) == 0:
                xs.append(x.clamp(0, 1).cpu())

        x = x.clamp(0, 1).cpu()
        
    return x, xs
    
def show_imgs(xs, cols=5):
    rows = math.ceil(len(xs) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = axes.flatten()

    for i in range(len(xs)):
        axes[i].imshow(xs[i][0].detach().cpu().numpy(), cmap='gray')
        axes[i].axis('off')

    # Ocultar ejes sobrantes si n no es múltiplo de cols
    for j in range(len(xs), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

ver el proceso de sampling

In [ ]:
# MNIST
xs = sample_raw_img(model, scheduler, image_size=(1, 28, 28), num_images=5)
show_imgs(xs)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (140x28 and 200x200)

ver el proceso de denoising paso a paso

In [ ]:
x = torch.randn(1, 1, 28, 28) # imagen de ruido inicialGE
x, xs = denoise_img(model, scheduler, x, num_steps=300, step_imgs=10)
show_imgs(xs)

# [EXTRA] Extract all VAE latents
Using the trained VAE, we can extract the latent representations of the audio samples in the dataset. This will allow us to train a diffusion model on these latent representations.

In [ ]:
from src.models import VAE
from src.dataset import NSynth
from tqdm import tqdm 

def load_vae(input_height=1500, input_width=251, latent_dim=200, channels=[2,16,32,64], model_path=r"C:\Users\Articuno\Desktop\TFG-info\data\models\vae.pth"):
    model = VAE((input_height, input_width), latent_dim=latent_dim, channels=channels).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    print("VAE Loaded")
    return model

def extract_latents(model, dataset, stft_transform):
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    all_z = []
    pbar = tqdm(dataloader, desc="Extracting latents")
    with torch.no_grad():
        for waveform, _, _, _ in pbar:
            waveform = waveform.to(device)
            stft_spec = stft_transform(waveform)
            # log_mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec)
            # x = torch.cat([log_mag, sin, cos], dim=1).to(device)
            log_mag, phase = compute_magnitude_and_phase(stft_spec)
            x = torch.cat([log_mag, phase], dim=1).to(device)
            feat, mu, logvar = model.encoder(x)
            z = mu # TODO no se lo que hace esto
            # z = model.reparameterize(mu, logvar) # TODO no se lo que hace esto
            all_z.append(z.cpu())
    all_z = torch.cat(all_z, dim=0)    
    return all_z    

In [ ]:
# extract_latents una sola vez
n_fft = 1500 # DISMINUIR TAMAÑO PARA OPTIMIZAR
hop_length = 250
win_length = n_fft

stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=None, onesided=False, center=False
).to(device)
latents = extract_latents(load_vae(), NSynth('training'), stft_transform)

# Calcula media y std del dataset de latentes
latent_mean = latents.mean()
latent_std = latents.std()
latents_norm = (latents - latent_mean) / latent_std  # que tengan distribución ~N(0,1)
torch.save({'mean': latent_mean, 'std': latent_std}, paths['latent']['training_norm_params'])
torch.save(latents_norm, paths['latent']['dataset_training'])

In [ ]:
import torch
# Mira cómo genera latentes tu VAE:
with torch.no_grad():
    vae = load_vae()
    z = vae.encode( NSynth('training'))
print(f"shape del latente original: {z.shape}")

VAE Loaded


AttributeError: 'VAE' object has no attribute 'encode'

Hay que normalizar las latentes antes de entrenar

In [ ]:
latents = torch.load(paths['latent']['dataset_training'])
print(f"shape: {latents.shape}")
print(f"mean: {latents.mean():.3f}")
print(f"std:  {latents.std():.3f}")
print(f"min:  {latents.min():.3f}")
print(f"max:  {latents.max():.3f}")

shape: torch.Size([289205, 200])
mean: 0.000
std:  1.000
min:  -17.184
max:  18.453


In [ ]:
latents = torch.load(paths['latent']['dataset_training'])
print(latents.shape)  # what does this print?

torch.Size([289205, 200])
